Load data

In [11]:
import pandas as pd

df = pd.read_csv("retail_sales_dataset.csv")
df.head()

,transaction_id,transaction_date,customer_id,customer_gender,customer_age_group,customer_segment,product_id,product_name,category,brand,quantity,unit_price,discount_pct,sales_amount,payment_method,sales_channel,region
0,T0000001,2024-04-24,C000820,Other,35-44,Returning,P1082,Dumbbells,Sports,Brand 1,2,313.53,20,501.65,Debit Card,Online,North
1,T0000002,2025-07-12,C002849,Other,45-54,New,P1087,Running Shoes,Sports,Brand 3,1,366.16,0,366.16,Credit Card,Online,South
2,T0000003,2025-06-01,C019727,Male,55+,Returning,P1030,Sneakers,Clothing,Brand 3,1,27.99,0,27.99,Gift Card,In-Store,South
3,T0000004,2025-08-26,C009116,Male,25-34,VIP,P1058,Sunscreen,Beauty,Brand 1,2,102.01,15,173.42,Cash,In-Store,South
4,T0000005,2024-12-10,C003350,Male,45-54,New,P1028,Sneakers,Clothing,Brand 1,1,259.55,0,259.55,Cash,In-Store,Central


Kiểm tra dữ liệu

In [12]:
df.shape
df.info()
df.describe()
df.isnull().sum()
df.duplicated().sum()

<class 'pandas.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 17 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   transaction_id      120000 non-null  str    
 1   transaction_date    120000 non-null  str    
 2   customer_id         120000 non-null  str    
 3   customer_gender     120000 non-null  str    
 4   customer_age_group  120000 non-null  str    
 5   customer_segment    120000 non-null  str    
 6   product_id          120000 non-null  str    
 7   product_name        120000 non-null  str    
 8   category            120000 non-null  str    
 9   brand               120000 non-null  str    
 10  quantity            120000 non-null  int64  
 11  unit_price          120000 non-null  float64
 12  discount_pct        120000 non-null  int64  
 13  sales_amount        120000 non-null  float64
 14  payment_method      120000 non-null  str    
 15  sales_channel       120000 non-null  str    


np.int64(0)

Phát hiện giá trị ngoại lai bằng phương pháp IQR

In [13]:
Q1 = df["sales_amount"].quantile(0.25)
Q3 = df["sales_amount"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df["sales_amount"] < lower_bound) | (df["sales_amount"] > upper_bound)]

print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Number of outliers:", len(outliers))
print("Outlier percentage:", round(len(outliers) / len(df) * 100, 2), "%")

Lower bound: -350.09999999999997
Upper bound: 948.46
Number of outliers: 8309
Outlier percentage: 6.92 %


Các giá trị ngoại lai của sales_amount được kiểm tra bằng phương pháp IQR. Kết quả cho thấy có 8,309 giá trị ngoại lai, chiếm 6.92% dữ liệu. Đây là các giao dịch có doanh số cao, nên chỉ phát hiện và giữ lại để tránh mất thông tin quan trọng.

Bài toán: Dự đoán doanh số bán hàng.
Target variable: sales_amount.
Loại bài toán: Regression.

Bỏ cột ID

In [14]:
df = df.drop(columns=["transaction_id", "customer_id", "product_id"])

Xử lý ngày tháng

In [15]:
df["transaction_date"] = pd.to_datetime(df["transaction_date"])

df["year"] = df["transaction_date"].dt.year
df["month"] = df["transaction_date"].dt.month
df["day"] = df["transaction_date"].dt.day
df["dayofweek"] = df["transaction_date"].dt.dayofweek

df = df.drop(columns=["transaction_date"])

Tách X và y

In [16]:
X = df.drop(columns=["sales_amount"])
y = df["sales_amount"]

Encode dữ liệu chữ

In [17]:
X = pd.get_dummies(X, drop_first=True)

Chia train/test

In [18]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

cell kiểm tra

In [19]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(96000, 74)
(24000, 74)
(96000,)
(24000,)


In [20]:
train_data = X_train.copy()
train_data["sales_amount"] = y_train

test_data = X_test.copy()
test_data["sales_amount"] = y_test

train_data.to_csv("dinh_train_data_80.csv", index=False)
test_data.to_csv("dinh_test_data_20.csv", index=False)